In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

matches = pd.read_csv("../data/matches.csv", parse_dates=["GAME_DATE"])

features = ["WIN_PCT_SEASON_HOME", "WIN_PCT_LAST10_HOME", "PM_LAST10_HOME", "REST_DAYS_HOME", "B2B_HOME",
            "WIN_PCT_SEASON_AWAY", "WIN_PCT_LAST10_AWAY", "PM_LAST10_AWAY", "REST_DAYS_AWAY", "B2B_AWAY"]
target = "HOME_WIN"

test_seasons = ["2024-25", "2025-26"]
train = matches[~matches["SEASON"].isin(test_seasons)]
test = matches[matches["SEASON"].isin(test_seasons)]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]
print("Entraînement :", len(train), "matchs | Test :", len(test), "matchs")

Entraînement : 10605 matchs | Test : 2418 matchs


In [2]:
baseline = y_test.mean()
print(f"Baseline (toujours prédire domicile) : {baseline:.1%}")

Baseline (toujours prédire domicile) : 55.0%


In [3]:
model = LogisticRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(f"Modèle : {accuracy_score(y_test, preds):.1%}")

Modèle : 66.5%


In [4]:
weights = pd.Series(model.coef_[0], index=features).sort_values()
weights

WIN_PCT_SEASON_AWAY   -1.861583
WIN_PCT_LAST10_HOME   -0.853155
B2B_HOME              -0.128119
PM_LAST10_AWAY        -0.037437
REST_DAYS_AWAY         0.020538
REST_DAYS_HOME         0.052133
PM_LAST10_HOME         0.056103
B2B_AWAY               0.354112
WIN_PCT_LAST10_AWAY    0.629708
WIN_PCT_SEASON_HOME    2.042968
dtype: float64

In [5]:
X_train[["WIN_PCT_SEASON_HOME", "WIN_PCT_LAST10_HOME", "PM_LAST10_HOME"]].corr().round(2)

,WIN_PCT_SEASON_HOME,WIN_PCT_LAST10_HOME,PM_LAST10_HOME
WIN_PCT_SEASON_HOME,1.00,0.83,0.76
WIN_PCT_LAST10_HOME,0.83,1.00,0.87
PM_LAST10_HOME,0.76,0.87,1.00


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_scaled = make_pipeline(StandardScaler(), LogisticRegression())
model_scaled.fit(X_train, y_train)
print(f"Modèle standardisé : {accuracy_score(y_test, model_scaled.predict(X_test)):.1%}")

pd.Series(model_scaled[-1].coef_[0], index=features).sort_values()

Modèle standardisé : 66.7%


WIN_PCT_SEASON_AWAY   -0.348807
PM_LAST10_AWAY        -0.253254
WIN_PCT_LAST10_HOME   -0.204629
B2B_HOME              -0.045669
REST_DAYS_AWAY         0.015228
REST_DAYS_HOME         0.036189
B2B_AWAY               0.148527
WIN_PCT_LAST10_AWAY    0.154132
PM_LAST10_HOME         0.376180
WIN_PCT_SEASON_HOME    0.384364
dtype: float64

In [7]:
def evaluate(cols):
    """Entraîne un modèle avec les colonnes données et renvoie sa précision sur le test."""
    m = make_pipeline(StandardScaler(), LogisticRegression())
    m.fit(X_train[cols], y_train)
    return accuracy_score(y_test, m.predict(X_test[cols]))

indicators = {
    "Bilan saison":  ["WIN_PCT_SEASON_HOME", "WIN_PCT_SEASON_AWAY"],
    "Forme récente": ["WIN_PCT_LAST10_HOME", "WIN_PCT_LAST10_AWAY"],
    "Domination":    ["PM_LAST10_HOME", "PM_LAST10_AWAY"],
}

print(f"Avec les 6 features      : {evaluate(features):.1%}")
for name, cols in indicators.items():
    remaining = [f for f in features if f not in cols]
    print(f"Sans {name:<14}      : {evaluate(remaining):.1%}")

Avec les 6 features      : 66.7%
Sans Bilan saison        : 65.1%
Sans Forme récente       : 66.0%
Sans Domination          : 65.8%


In [8]:
proba = model_scaled.predict_proba(X_test)[:, 1]   # probabilité que l'équipe à domicile gagne

results = test[["GAME_DATE", "TEAM_ABBREVIATION_HOME", "TEAM_ABBREVIATION_AWAY", "HOME_WIN"]].copy()
results["PROBA_HOME"] = proba.round(2)
results.head(10)

,GAME_DATE,TEAM_ABBREVIATION_HOME,TEAM_ABBREVIATION_AWAY,HOME_WIN,PROBA_HOME
10605,2024-10-25,ATL,CHA,1,0.57
10606,2024-10-25,LAL,PHX,1,0.64
10607,2024-10-25,POR,NOP,0,0.03
10608,2024-10-25,TOR,PHI,1,0.30
10609,2024-10-25,ORL,BKN,1,0.94
10610,2024-10-25,CLE,DET,1,0.97
10611,2024-10-25,HOU,MEM,1,0.21
10612,2024-10-25,MIL,CHI,0,0.94
10613,2024-10-25,NYK,IND,1,0.08
10614,2024-10-25,UTA,GSW,0,0.08


In [9]:
from sklearn.metrics import brier_score_loss

baseline_proba = [y_train.mean()] * len(y_test)   # toujours dire "environ 57 %" pour l'équipe à domicile
print(f"Brier baseline : {brier_score_loss(y_test, baseline_proba):.4f}")
print(f"Brier modèle   : {brier_score_loss(y_test, proba):.4f}")

Brier baseline : 0.2478
Brier modèle   : 0.2120


In [10]:
results["TRANCHE"] = pd.cut(results["PROBA_HOME"], bins=[0, .3, .4, .5, .6, .7, .8, 1])
results.groupby("TRANCHE", observed=True)["HOME_WIN"].agg(["count", "mean"]).round(2)

,count,mean
TRANCHE,,
"(0.0, 0.3]",210,0.22
"(0.3, 0.4]",260,0.30
"(0.4, 0.5]",403,0.44
"(0.5, 0.6]",483,0.52
"(0.6, 0.7]",518,0.66
"(0.7, 0.8]",353,0.79
"(0.8, 1.0]",191,0.83
